# BIOMASS stack: first inspection

Goal: verify the available stack folders and metadata **before loading any SLC arrays**.

This starter assumes a local `config/local.toml` has been created. The supplied
BIOMASS loader expects a specific primary/reference annotation and product order;
check that ordering before running `bmStack(...)`. The study stack reportedly has
7 acquisition folders, with the middle acquisition as the master.

In [ ]:
import os
print("Kernel PID:", os.getpid())

In [ ]:
from pathlib import Path
from biomass_tomosar.paths import biomass_stack_dir

root = biomass_stack_dir()
products_by_name = sorted(p for p in root.glob("BIO*") if p.is_dir())
assert products_by_name, f"No BIOMASS product folders found under {root}"

for i, product in enumerate(products_by_name):
    prefix = product.name[:70].lower()
    primary_xml = product / "annotation_primary" / f"{prefix}_annot.xml"
    coreg_xml = product / "annotation_coregistered" / f"{prefix}_annot.xml"
    amp_tif = product / "measurement" / f"{prefix}_i_abs.tiff"
    phase_tif = product / "measurement" / f"{prefix}_i_phase.tiff"
    print(f"[{i}] {product.name}")
    print(f"    primary={primary_xml.is_file()} coreg={coreg_xml.is_file()} "
          f"amplitude={amp_tif.is_file()} phase={phase_tif.is_file()}")

candidates = [
    p for p in products_by_name
    if (p / "annotation_primary" / f"{p.name[:70].lower()}_annot.xml").is_file()
]
print("Primary-annotation candidates:", [p.name for p in candidates])

## Example from script
This example runs, but loads everything at once, several GB of RAM USED! Don't run it like this, load less things. Not robust, very prone to crashes 
It also uses the SWAP memory once RAM is full
## Useful commands
free - h
watch -n 1 free -h


In [ ]:
from glob import glob
import matplotlib.pyplot as plt
from biomass_tomosar.bm_data_loader import bmStack
import numpy as np
from tqdm.auto import tqdm
# Gets paths to BIOMASS STA products
baseDirs = np.sort(glob("../data/BIOMASS_STACK/BIO*"))
print(baseDirs)

# Sets up bmStack object
stack = bmStack(baseDirs)
# Usefull bmStack variables
# stack.Ntracks : number of tracks
# stack.rg : Slant range of the primary image [m]
# stack.az : Azimuth of the primary image [m]
# stack.nli : Number of azimuth lines
# stack.nsa : Number of range samples

# Imports SLCS without terrain flattening    
# stack.importSlcs loads the BIOMASS SLCS into memory as a numpy array of size (N_tracks, N_pol, N_az, N_rg),
# with N_pol = 4, and the polarisations ordered as (HH, HV, VH, VV).
# terrain_flatten toggles whether or not terrain flattening is applied to the stack during import
# swap_axes toggles whether or not the range and azimuth axes should be swaped.
# When loaded, slcs can be retrieved through stack.slcs
stack.importSlcs(terrain_flatten=False, swapaxes=False)

# Displays polarimetric SLCs for in dB
pols = ["HH","HV","VH","VV"]
_,ax = plt.subplots(4,stack.Ntracks,layout="constrained",figsize=(9,12))
dB_imgs = 20*np.log10(np.abs(stack.slcs[:,:,::9,::27]))
for i in range(ax.shape[0]):
    for j in range(ax.shape[1]):
        p = ax[i,j].imshow(dB_imgs[j,i],cmap="gray",vmin=-27,vmax=10,extent=[1e-3*stack.rg[0],1e-3*stack.rg[-1],1e-3*stack.az[-1],1e-3*stack.az[0]])
# [ax[0,j].set_title(os.path.basename(stack.baseDirs[j])) for j in range(stack.Ntracks)]
[ax[i,0].set_ylabel(pols[i] + "\nAzimuth [km]") for i in range(len(pols))]
[axi.set_xlabel("Range [km]") for axi in ax[-1,:]]
[axi.set_xticklabels("") for axi in ax[:-1,:].flatten()]
[axi.set_yticklabels("") for axi in ax[:,1:].flatten()]
plt.colorbar(p,ax=ax[-1,:],label="Beta0 [dB]",pad=0.01,orientation="horizontal")
plt.show()

# Imports LUT variables
# stack.lut.get() takes up to two arguments, the first a being netcdf group or variable, the second being a variable
# in the group. stack.lut.get().nc returns the corresponding netcdf object
# If the variable is an array, stack.lut.get().data returnes the variable as a numpy array, as it is stored in the LUT.
# This array is downsampled w.r.t. the full image size
# stack.lut.get().upsampled upsamples the array to the full image size and returns it as a numpy array
#
# LUT groups and variables can also be found in the BIOMASS l1C products description
# Prints netCDF4 dataset for image 0
stack.luts[0].get().nc

# Prints variables for the group "coregistation"
print(stack.luts[0].get("coregistration").nc.variables)

# Gets upsampled terrain flattening phases
flat_phase_upsamp = np.array([lut.get("coregistration","flatteningPhaseScreen").upsampled for lut in tqdm(stack.luts,desc="Upsampling flattening phases")])
print(flat_phase_upsamp.shape)


## Test Cells To play with the loaded Data



### Loaded Data Verification


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

print("Kernel PID:", os.getpid())
print("Acquisitions:", stack.Ntracks)
print("SLC shape:", stack.slcs.shape)
print("Data type:", stack.slcs.dtype)
print(f"SLC memory: {stack.slcs.nbytes / 2**30:.2f} GiB")
with open("/proc/self/status") as f:
    for line in f:
        if line.startswith(("VmRSS:", "VmSwap:", "VmSize:")):
            print(line.strip())
# Creates a reference, NOT a copy
slcs = stack.slcs

print("Same memory:", np.shares_memory(slcs, stack.slcs))

### Small Region Plot


In [ ]:
MASTER = 3
SLAVE = 0
POL = 0
SIZE = 512

# Select a patch near the image center
height, width = slcs.shape[-2:]

y0 = (height - SIZE) // 2
x0 = (width - SIZE) // 2

master = slcs[MASTER, POL, y0:y0+SIZE, x0:x0+SIZE]
slave = slcs[SLAVE, POL, y0:y0+SIZE, x0:x0+SIZE]

print("Master:", master.shape)
print("Slave:", slave.shape)

print("Master shares memory:", np.shares_memory(master, slcs))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, image, title in zip(
    axes,
    [master, slave],
    ["Master (acquisition 4)", "Slave (acquisition 1)"]
):
    amplitude_db = 20 * np.log10(np.maximum(np.abs(image), 1e-8))

    im = ax.imshow(amplitude_db, cmap="gray", vmin=-30, vmax=10)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label="Amplitude [dB]")

plt.tight_layout()
plt.show()

In [ ]:
interferogram = master * np.conj(slave)
phase = np.angle(interferogram)

fig, ax = plt.subplots(figsize=(7, 6))

im = ax.imshow(
    phase,
    cmap="twilight",
    vmin=-np.pi,
    vmax=np.pi
)

ax.set_title("Wrapped interferometric phase (4 vs 1)")
fig.colorbar(im, ax=ax, label="Phase [rad]")

plt.show()


In [ ]:
from scipy.ndimage import uniform_filter

window = 9

cross = master * np.conj(slave)

cross_mean = (
    uniform_filter(cross.real, size=window)
    + 1j * uniform_filter(cross.imag, size=window)
)

power_m = uniform_filter(np.abs(master)**2, size=window)
power_s = uniform_filter(np.abs(slave)**2, size=window)

coherence = (
    np.abs(cross_mean)
    / np.sqrt(np.maximum(power_m * power_s, 1e-20))
)

coherence = np.clip(coherence, 0, 1)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(coherence, cmap="viridis", vmin=0, vmax=1)

ax.set_title("Local interferometric coherence")
fig.colorbar(im, ax=ax, label="Coherence magnitude")

plt.show()

## SKP Phases Check

In [ ]:
lut = stack.luts[0]

print(lut.get().nc.groups.keys())

print(
    lut.get("skpPhaseCalibration").nc.variables.keys()
)

for i, lut in enumerate(stack.luts):
    groups = lut.get().nc.groups
    skp = groups.get("skpPhaseCalibration")

    print(f"Acquisition {i+1}:")
    if skp is not None:
        for name, var in skp.variables.items():
            print(f"  {name}: {var.shape}")
    else:
        print("  SKP group not present")

### Vizualize SKP

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

lut = stack.luts[0]

phase_var = lut.get(
    "skpPhaseCalibration",
    "skpCalibrationPhaseScreen"
).nc

quality_var = lut.get(
    "skpPhaseCalibration",
    "skpCalibrationPhaseScreenQuality"
).nc

# Keep the NetCDF no-data mask
phase = np.ma.masked_invalid(phase_var[:])
quality = np.ma.masked_invalid(quality_var[:])

print("Phase shape:", phase.shape)
print("Quality shape:", quality.shape)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

im1 = axes[0].imshow(
    phase,
    cmap="twilight",
    vmin=-np.pi,
    vmax=np.pi,
    aspect="auto"
)
axes[0].set_title("SKP calibration phase [rad]")
fig.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(
    quality,
    cmap="viridis",
    vmin=0,
    vmax=1,
    aspect="auto"
)
axes[1].set_title("SKP estimation quality")
fig.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

In [ ]:
print("Valid phase pixels:", phase.count())
print("Phase range:", phase.min(), phase.max())
print("Mean quality:", quality.mean())

In [ ]:
flags = [
    "skpPhaseCalibrationFlag",
    "skpPhaseCorrectionFlag",
    "skpPhaseCorrectionFlatteningOnlyFlag",
]

for i, annot in enumerate(stack.annot_coreg):
    print(f"\nAcquisition {i+1}")

    for flag in flags:
        values = annot.xpath(
            f"//*[local-name()='{flag}']/text()"
        )
        print(f"  {flag}: {values[0] if values else 'Not found'}")